# **세팅설정**

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k, stride):
        super().__init__()
        self.conv = nn.Conv1d(in_ch, out_ch, k, stride=stride, padding=k // 2)
        self.act  = nn.SiLU()

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.act(self.conv(x))
        x = x.permute(0, 2, 1)
        return x

from abc import ABC, abstractmethod

class RotaryEmbedding(nn.Module):
    def __init__(self, d_head, max_t=2048):
        super().__init__()
        assert d_head % 2 == 0
        inv = 1 / (10000 ** (torch.arange(0, d_head, 2).float() / d_head))
        f = torch.outer(torch.arange(max_t).float(), inv)
        self.register_buffer("cos", f.cos().to(torch.float16), persistent=False)
        self.register_buffer("sin", f.sin().to(torch.float16), persistent=False)
    def forward(self, x):
        T = x.size(-2)
        a, b = x.chunk(2, dim=-1)
        cos = self.cos[:T].to(dtype=x.dtype)
        sin = self.sin[:T].to(dtype=x.dtype)
        return torch.cat((
            torch.addcmul(a * cos, b, sin, value=-1),
            torch.addcmul(b * cos, a, sin)
        ), dim=-1)

class BaseAttention(nn.Module, ABC):
    def __init__(self, d_model, num_heads, rope):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.out = nn.Linear(d_model, d_model)
        self.rope = rope
    def sdpa(self, Q, K, V):
        return self.out(F.scaled_dot_product_attention(Q, K, V, dropout_p=0.1 if self.training else 0.0).transpose(1, 2).flatten(2))
    def vp(self, x):
        B, T, C = x.shape
        N = C//self.d_model
        return x.view(B, T, N, self.num_heads, self.d_head).permute(2, 0, 3, 1, 4).unbind(0)
    @abstractmethod
    def forward(self, x):
        pass

class SelfAttention(BaseAttention):
    def __init__(self, d_model, num_heads, rope):
        super().__init__(d_model, num_heads, rope)
        self.qkv = nn.Linear(d_model, d_model * 3)
    def forward(self, x):
        Q, K, V = self.vp(self.qkv(x))
        Q, K = self.rope(Q), self.rope(K)
        return self.sdpa(Q, K, V)

class WindowSelfAttention(BaseAttention):
    def __init__(self, d_model, num_heads, rope, group_size=64, context_size=32):
        super().__init__(d_model, num_heads, rope)
        self.group_size = group_size
        self.context_size = context_size
        self.attention_size = group_size + context_size * 2
        self.qkv = nn.Linear(d_model, d_model * 3)

    def _window(self, x, tail=0):
        N = self.context_size
        x = torch.cat((x[:, :, -N:], x, x[:, :, :N + tail]), dim=2)
        return x.unfold(2, self.attention_size, self.group_size).permute(0, 2, 1, 4, 3)

    def forward(self, x):
        B, T, C = x.shape
        G = self.group_size
        H, D = self.num_heads, self.d_head

        Q, K, V = self.vp(self.qkv(x))
        Q, K = self.rope(Q), self.rope(K)

        if T <= self.attention_size:
            return self.sdpa(Q, K, V)

        tail = (-T) % G

        if tail:
            Q = torch.cat((Q, Q[:, :, :tail]), dim=2)

        Q = Q.view(B, H, -1, G, D).permute(0, 2, 1, 3, 4)
        K = self._window(K, tail)
        V = self._window(V, tail)

        out = F.scaled_dot_product_attention(Q, K, V,dropout_p=0.1 if self.training else 0.0)
        out = out.permute(0, 1, 3, 2, 4).reshape(B, -1, C)[:, :T]
        return self.out(out)

class CrossAttention(BaseAttention):
    def __init__(self, d_model, num_heads, rope, num_queries):
        super().__init__(d_model, num_heads, rope)
        self.num_queries = num_queries
        self.query = nn.Parameter(torch.randn(1, num_heads, num_queries, self.d_head) * 0.02)
        self.kv = nn.Linear(d_model, d_model * 2)

    def forward(self, x):
        Q = self.query.expand(x.size(0), -1, -1, -1)
        K, V = self.vp(self.kv(x))
        Q, K = self.rope(Q), self.rope(K)
        return self.sdpa(Q, K, V)

class BaseTransformerBlock(nn.Module):
    def __init__(self, d_model, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * mlp_ratio),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * mlp_ratio, d_model),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class SelfTransformerBlock(BaseTransformerBlock):
    def __init__(self, d_model, num_heads, rope, mlp_ratio=4, dropout=0.1):
        super().__init__(d_model, mlp_ratio, dropout)
        self.attn = SelfAttention(d_model, num_heads, rope)
        self.norm3 = nn.LayerNorm(d_model)
        self.norm4 = nn.LayerNorm(d_model)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        x = x + self.attn(self.norm3(x))
        x = x + self.ffn(self.norm4(x))
        return x

class WindowTransformerBlock(BaseTransformerBlock):
    def __init__(self, d_model, num_heads, rope, group_size=64, context_size=32, mlp_ratio=4, dropout=0.1):
        super().__init__(d_model, mlp_ratio, dropout)
        self.attn = WindowSelfAttention(
            d_model, num_heads, rope,
            group_size=group_size,
            context_size=context_size
        )

class CrossTransformerBlock(BaseTransformerBlock):
    def __init__(self, d_model, num_heads, rope, num_queries, mlp_ratio=4, dropout=0.1):
        super().__init__(d_model, mlp_ratio, dropout)
        self.attn = CrossAttention(d_model, num_heads, rope, num_queries)
    def forward(self, x):
        x = self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class LinearBlock(nn.Module):
    def __init__(self, in_f, out_f, act=True, dropout=0.0):
        super().__init__()
        self.fc   = nn.Linear(in_f, out_f)
        self.act  = nn.GELU() if act else nn.Identity()
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
    def forward(self, x):
        return self.drop(self.act(self.fc(x)))


class CA1DEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(256, 64, padding_idx=0)
        rope16  = RotaryEmbedding(16,  16384)
        rope32  = RotaryEmbedding(32,  8192)
        rope64  = RotaryEmbedding(64,  2048)
        self.CA = nn.Sequential(
            nn.Dropout(0.1),
            WindowTransformerBlock(64,  4, rope16, group_size=96, mlp_ratio=2),
            ConvBlock(64, 128, 5, 2),
            WindowTransformerBlock(128, 4, rope32, group_size=64, mlp_ratio=2),
            ConvBlock(128, 256, 5, 2),
            WindowTransformerBlock(256, 8, rope32, group_size=64, mlp_ratio=3),
            ConvBlock(256, 512, 5, 2),
        )
        self.ATTN = nn.Sequential(
            nn.Dropout(0.1),
            SelfTransformerBlock(512, 8,  rope64),
            SelfTransformerBlock(512, 16, rope32),
            SelfTransformerBlock(512, 16, rope32),
            SelfTransformerBlock(512, 16, rope32),
            LinearBlock(512, 256),
            SelfTransformerBlock(256, 8,  rope32),
            SelfTransformerBlock(256, 16, rope16),
            SelfTransformerBlock(256, 16, rope16),
        )
        self.norm = nn.LayerNorm(256)

    def forward(self, x):
        x = x.squeeze(1).long()
        x = self.embedding(x)
        x = self.CA(x)
        x = self.ATTN(x)
        return self.norm(x)


class CA1D(nn.Module):
    def __init__(self, num_classes, encoder=None):
        super().__init__()
        self.encoder = encoder if encoder is not None else CA1DEncoder()
        self.cross_attn = CrossTransformerBlock(256, 16, RotaryEmbedding(16,  2048), 8, mlp_ratio=2)
        self.flatten = nn.Flatten()
        self.drop = nn.Dropout(0.1)
        fc_cfg = [
            # in,  out,        act,   dropout
            (2048, 64,        True,  0.4),
            (64,   num_classes,False, 0.0),
        ]
        self.head = nn.Sequential(*[LinearBlock(*c) for c in fc_cfg])

    def forward(self, x):
        return self.head(self.drop(self.flatten(self.cross_attn(self.encoder(x)))))

In [2]:
choseong = (
    "ㄱ", "ㄲ", "ㄴ", "ㄷ", "ㄸ", "ㄹ", "ㅁ",
    "ㅂ", "ㅃ", "ㅅ", "ㅆ", "ㅇ", "ㅈ", "ㅉ",
    "ㅊ", "ㅋ", "ㅌ", "ㅍ", "ㅎ",
)

jungseong = (
    "ㅏ", "ㅐ", "ㅑ", "ㅒ", "ㅓ", "ㅔ", "ㅕ",
    "ㅖ", "ㅗ", "ㅘ", "ㅙ", "ㅚ", "ㅛ", "ㅜ",
    "ㅝ", "ㅞ", "ㅟ", "ㅠ", "ㅡ", "ㅢ", "ㅣ",
)

jongseong = (
    "",
    "ㄱ", "ㄲ", "ㄳ", "ㄴ", "ㄵ", "ㄶ",
    "ㄷ", "ㄹ", "ㄺ", "ㄻ", "ㄼ", "ㄽ",
    "ㄾ", "ㄿ", "ㅀ", "ㅁ", "ㅂ", "ㅄ",
    "ㅅ", "ㅆ", "ㅇ", "ㅈ", "ㅊ", "ㅋ",
    "ㅌ", "ㅍ", "ㅎ",
)

char_to_id = {
    **{str(i): i + 1 for i in range(10)},
    **{chr(ord("A") + i): 11 + i for i in range(26)},
    **{chr(ord("a") + i): 37 + i for i in range(26)},

    **{j:i+63 for i, j in enumerate(choseong)},
    "ㄳ": 82, "ㄵ": 83, "ㄶ": 84, "ㄺ": 85, "ㄻ": 86,
    "ㄼ": 87, "ㄽ": 88, "ㄾ": 89, "ㄿ": 90, "ㅀ": 91,
    "ㅄ": 92,
    **{j:i+93 for i, j in enumerate(jungseong)},

    **{j:114+i for i, j in enumerate([" ", "!", '"', "#", "$", "%", "&", "'","(", ")", "*", "+", ",", "-", ".", "/",":", ";", "<", "=", ">", "?", "@", "[","\\", "]", "^", "_", "`", "{", "|", "}","~", "\n", "\t"])}
}

frequent_tokens = (
    "and","be","at","com","de","do","ed","en","er","for","have","hi","ing","ion","in","it","is","no","other","of","or","the","th","to","use","up"," thi"," is"," a"," i",
    "가","기","니다","는","그","고","나","다","도","라","를","면","서","수","시","아니","안","이다","은","을","으로","이","에","에서","의","좋","지","한","함","때문",
    "「","」","『","』","♡","♥","※","☆","★","□","■","○","●","⦁",
    "😂", "😊", "😍", "😭", "😱",
    "👍", "❤️", "🔥", "🎉", "🙏","✨","📝","♻️",
    "💢", "❓", "❗", "‼️", "⁉️", "💡",
)
TOKEN_END = None

token_trie = {}

for token_id, token in enumerate(frequent_tokens, 149):
    node = token_trie

    for char in token:
        node = node.setdefault(char, {})

    node[TOKEN_END] = token_id

END_ID = 0
UNKNOWN_ID = 255
def encode_text(text: str) -> list[int]:
    encoded = []

    i=0
    l=len(text)
    while i < l:
        node = token_trie
        j = i

        matched_id = None
        matched_end = i

        while j < l:
            node = node.get(text[j])

            if node is None:
                break

            j += 1

            token_id = node.get(TOKEN_END)
            if token_id is not None:
                matched_id = token_id
                matched_end = j

        if matched_id is not None:
            encoded.append(matched_id)
            i = matched_end
            continue

        # ______________________________________________________
        char = text[i]
        code = ord(char)
        if 0xAC00 <= code <= 0xD7A3:
            index = code - 0xAC00

            cho_index = index // 588
            jung_index = (index % 588) // 28
            jong_index = index % 28

            encoded.append(char_to_id[choseong[cho_index]])
            encoded.append(char_to_id[jungseong[jung_index]])

            if jong_index != 0:
                encoded.append(char_to_id[jongseong[jong_index]])

        # 등록된 일반 문자
        elif char in char_to_id:
            encoded.append(char_to_id[char])

        # 미등록 유니코드 문자
        else:
            encoded.append(UNKNOWN_ID)
            hex_code=format(code, "X")
            encoded.extend(char_to_id[hex_char]for hex_char in hex_code)
            encoded.append(char_to_id[";"])

        i+=1

    return encoded

# **파인튜닝**

In [ ]:
# [4/4] 파인튜닝 — encoder_pretrained.pth 로드 후 commits_t.csv로 분류 학습
import joblib
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report

FT_EPOCHS = 10
FT_HEAD_LR = 3e-4       # 새로 초기화되는 head (기존 lr 그대로)
FT_ENCODER_LR = 2e-5    # 사전학습 인코더는 1/10. 3e-4로 두면 기존과 동일


class DatasetBYTETEXT(Dataset):
    def __init__(self, raw_data_list, labels):
        self.x_data = [
            torch.tensor(encode_text(item), dtype=torch.uint8)
            for item in raw_data_list
        ]
        self.y_data = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):
        return self.x_data[idx], self.y_data[idx]


def collate_fn(batch):
    xs = [item[0] for item in batch]
    ys = [item[1] for item in batch]

    xs_padded = pad_sequence(xs, batch_first=True, padding_value=0)
    xs_padded = xs_padded.unsqueeze(1)

    ys_tensor = torch.stack(ys)
    return xs_padded, ys_tensor


class EMA:
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        self.buf_shadow = {}
        self.buf_backup = {}
        self._register()

    def _register(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()
        for name, buf in self.model.named_buffers():
            self.buf_shadow[name] = buf.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = (
                    self.decay * self.shadow[name] + (1 - self.decay) * param.data
                )
        for name, buf in self.model.named_buffers():
            if torch.is_floating_point(buf):
                self.buf_shadow[name] = (
                    self.decay * self.buf_shadow[name] + (1 - self.decay) * buf.data
                )
            else:
                self.buf_shadow[name] = buf.data.clone()

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name]
        for name, buf in self.model.named_buffers():
            self.buf_backup[name] = buf.data.clone()
            buf.data = self.buf_shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]
        self.backup = {}
        for name, buf in self.model.named_buffers():
            buf.data = self.buf_backup[name]
        self.buf_backup = {}

    def state_dict(self):
        return {k: v.clone() for k, v in self.shadow.items()}

    def load_state_dict(self, state):
        self.shadow = {k: v.clone() for k, v in state.items()}


def main_finetune():
    torch.set_float32_matmul_precision('high')

    data_dict = pd.read_csv("annotated_dataset.csv")

    data_dict['masked_commit_message'] = data_dict['masked_commit_message'].str.strip()

    raw_records = data_dict.to_dict('records')

    data_list = [
        {
            'data': f"{str(item["masked_commit_message"])}\n-------\n{item['git_diff'].splitlines()[0]}",
            'label': item["annotated_type"]
        }
        for item in raw_records
    ]


    x_list = [item['data'] for item in data_list]
    y_list = [item['label'] for item in data_list]

    encoder_le = LabelEncoder()
    y_encoded = encoder_le.fit_transform(y_list)
    joblib.dump(encoder_le, 'encoder.pkl')

    x_train, x_test, y_train, y_test = train_test_split(
        x_list, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
    )
    x_train, x_vel, y_train, y_vel = train_test_split(
        x_train, y_train, test_size=0.125, random_state=42, stratify=y_train
    )

    train_dataset = DatasetBYTETEXT(x_train, y_train)
    vel_dataset = DatasetBYTETEXT(x_vel, y_vel)
    test_dataset = DatasetBYTETEXT(x_test, y_test)

    train_loader = DataLoader(
        train_dataset, batch_size=16, shuffle=True,
        collate_fn=collate_fn, num_workers=2, pin_memory=True
    )
    vel_loader = DataLoader(
        vel_dataset, batch_size=64, shuffle=False,
        collate_fn=collate_fn, num_workers=2, pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=64, shuffle=False,
        collate_fn=collate_fn, pin_memory=True
    )

    # --- [모델 초기화] 사전학습 인코더 로드 <- 기존 지도학습과 다른 유일한 지점 ---
    pre_encoder = CA1DEncoder()
    pre_encoder.load_state_dict(torch.load('encoder_pretrained.pth', map_location=device))
    model = CA1D(num_classes=len(encoder_le.classes_), encoder=pre_encoder).to(device)
    print("사전학습 인코더 로드 완료: encoder_pretrained.pth")

    # model.compile()

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW([
        {'params': model.encoder.parameters(), 'lr': FT_ENCODER_LR},
        {'params': model.head.parameters(),    'lr': FT_HEAD_LR},
    ])
    ema = EMA(model, decay=0.99)

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FT_EPOCHS, eta_min=1e-6)

    print("--- 파인튜닝 시작 ---")
    for epoch in range(FT_EPOCHS):
        model.train()
        running_loss = 0.0

        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            ema.update()

            running_loss += loss.item() * batch_x.size(0)

        scheduler.step()

        model.eval()
        ema.apply_shadow()

        correct = 0
        with torch.no_grad():
            for batch_x, batch_y in vel_loader:
                outputs = model(batch_x.to(device))
                _, predicted = torch.max(outputs, 1)
                correct += (predicted == batch_y.to(device)).sum().item()

        ema.restore()

        epoch_loss = running_loss / len(train_dataset)
        val_acc = correct / len(vel_dataset) * 100
        print(f"Epoch {epoch+1}/{FT_EPOCHS} - Loss: {epoch_loss:.4f} | Val Acc: {val_acc:.2f}%")

    print("\n--- 모델 검증 시작 ---")
    model.eval()
    ema.apply_shadow()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    print("[혼동 행렬 (Confusion Matrix)]")
    cm = confusion_matrix(all_labels, all_preds)
    cm_df = pd.DataFrame(cm, index=encoder_le.classes_, columns=encoder_le.classes_)
    display(cm_df)
    print("\n[정밀도 및 재현율 (Classification Report)]")
    report_dict = classification_report(all_labels, all_preds,
                                        target_names=encoder_le.classes_, output_dict=True)
    report_df = pd.DataFrame(report_dict).transpose()
    pd.set_option('display.float_format', '{:.4f}'.format)
    display(report_df)

    test_sentence = 'remove control function bug'
    single_input = torch.tensor(encode_text(test_sentence), dtype=torch.uint8)
    single_input = single_input.unsqueeze(0).to(device)

    with torch.no_grad():
        pred_output = model(single_input)
        _, pred_class = torch.max(pred_output, 1)
        print(f"\n[실시간 추론] '{test_sentence}' -> 예측 클래스: {encoder_le.classes_[pred_class.item()]}")

    torch.save(model.state_dict(), 'model_finetuned.pth')
    print("저장 완료: model_finetuned.pth")


if __name__ == "__main__":
    main_finetune()